# Number Cross — Jane Street, August 2014

https://www.janestreet.com/puzzles/number-cross-index/

## Solution

**407358** — the sum of the 5-digit entries.

![The finished grid](finished-grid.jpg)

## AI Use Disclaimer

I'm gonna try to solve this one by hand.  Used Python for some helper functions and to enumberate candidates at the end, but no AI use.

## Conclusion

Wow -- I loved this puzzle.  It felt like a cross between sudoku with the logic of the unique digits per clue, crossword puzzles, and number theory.  I had a blast with so many of the deductions, although the end was just kind of brute forcing candidates.  Certainlly was useful to know the divisibility rules for 3, 9 and 11 (which I had to look up).

## The puzzle

> Place the digits 1 thru 9 (no zeroes) in the crossword grid below so that all of the clues are
> satisfied. No digit is repeated in any one grid entry (e.g. 1-across is a grid entry), and no grid
> entry is used more than once within the puzzle.
>
> Submit the sum of the 8 five-digit grid entries in the completed puzzle.

![The puzzle grid and clues](number-cross.png)

## 1. Three-digit squares -- 11 clues!

In [35]:
three_digit_squares = [i**2 for i in range(10, 32) if len(set(str(i**2))) == 3]

no_repeating_digits = [n for n in three_digit_squares if len(set(str(n))) == 3]

print(no_repeating_digits)

[169, 196, 256, 289, 324, 361, 529, 576, 625, 729, 784, 841, 961]


The set of 3 digit squares is the most compelling starting place.  There are 11 of them and only 13 candidates once you enforce the no digit repeats rule.  

Candidates: [169, 196, 256, 289, 324, 361, 529, 576, 625, 729, 784, 841, 961]

1 accross and 1 down share a leading number, so they have to start with 1, 2, 3, 5, or 7 to be unique.  13 across is the smallest number in the grid, and has to start with a 1 or a 2 because, at minimum, 256 is in the grid as a 3 digit square (if 169 and 196 are left out). Thus, 1 down must end in a 1 or a 2, restricting it to exactly 361, and 1 across is 324.  We remove those from the candidates for other squares since we can't repeat numbers.

The next place to look is 7 across, which is a 3 digit square that begins 2 other 3 digit squares moving down. That helps reduce candidates.

Starting Candidates: [169, 196, 256, 289, 529, 576, 625, 729, 784, 841, 961]

Candidates reduced: [169, 196, 256, 289, 529, 576, 729, 784]

12 across is also a 3 digit square.  So the two downward squares from 7 and 8 down must have second digits that start a different square.  That is super restrictive.

Reduced Candidate set for 7 across: [529, 576]

This set of 4 squares forms a family of the numbers 529, 576, 784, 289, which you can arrange in 2 ways.  

At this point, I wrote some helper functions to generate candidates as we move through the solve.  

## 2. Helper functions

In [19]:
def find_primes(low, high):
    primes = []
    for n in range(low, high + 1):
        if is_prime(n):
            primes.append(n)
    return primes


def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True


def multiples_of(x, min, max):
    least_multiple = (min + x - 1) // x * x
    for i in range(least_multiple, max + 1, x):
        yield i


def filter_for_distinct(nums):
    for num in nums:
        digits = str(num)
        if "0" in digits:
            continue
        if len(digits) == len(set(digits)):
            yield num

Using our helper functions and the two possible ways to arrange the squares in 7/8 down and 7/12 across, we restrict 9 down to just 3 options.

In [20]:
print(find_primes(940, 949))
print(find_primes(690, 699))

[941, 947]
[691]


9 down is prime, and can be led by 94 or 69.

Candidates: [941, 947, 691]

However, 14 across must have digits summing to 42 using 7 distinct digits!  This is a sodoku style restraint, and means we must exclude 1 and 2 form the digits of 14.  Thus, nine down 947 less we introduce a 1 to 14 across.  This now also resolves 7/8 down and 7/12 across!

## 3. 4-across, 11-across and 5-down

Remaining 3 digit square square candidates: [169, 196, 256, 625, 729, 841, 961]

Now I will look at 4 and 11 across.  4 down is restricted to digits '1,2,3,4,6' and 6 down '5,6,7,8,9'.

So the valid 4 and 11 across squares are [169, 196, 256, 625]

Leading numbers possible for 5 down are 69, 65, 62, 96, 95, 92, 56, 59, 52, 26, 29, 25.

In [39]:
print(list(filter_for_distinct(multiples_of(947, 1000, 10000))))

[1894, 2841, 4735, 5682, 8523]


Cross referencing the possible leading number for 5 down with its clue give us only 1 valid solution: [5682]!  This gives us 4 across is 256 and 11 accros is 169.  

## 6. 39-across

In [25]:
find_primes(450, 500)

[457, 461, 463, 467, 479, 487, 491, 499]

39 accross is either 491 * 2 = 982 or 487 * 2 = 974

## 7. 29-across and 35-down

After a chunk of solving that I didn't write down since it was all on the paper, 29 across is pretty restricted by the 4 in second position, the fact that it's a multiple of 256, and the distinct digit rule.  Lets figure out how many candidates we have.  Additionally, the second digit of 35 down must be 7 or 8, further restricting 29

In [27]:
candidates_29 = list(filter_for_distinct(list(multiples_of(256, 100000, 256 * 987))))

candidates_29 = [candidate for candidate in candidates_29 if str(candidate)[1] == "4"]


print(candidates_29)

candidate_35 = [candidate / 256 for candidate in candidates_29]

print(candidate_35)

[143872, 147968, 148736, 243968, 248576, 249856]
[562.0, 578.0, 581.0, 953.0, 971.0, 976.0]


The only valid candidates for 35 down are 578 and 976, which makes the only valid candidates for 29 across 147968 and 249856.

This finding forces a 7 in the second positions of 35 down and 39 accross, forcing a 4 in second position of 36 down, giving us 36 down as 841.

Progress!!

## 8. 18-across and 31-down

In [28]:
print(str(69578 + 125))
print(str(69578 + 126))
print(str(69578 + 135))
print(str(69578 + 136))
print(str(69578 + 145))
print(str(69578 + 146))
print(str(69578 + 156))
print("________________")
print(str(69578 + 235))
print(str(69578 + 236))
print(str(69578 + 245))
print(str(69578 + 246))
print(str(69578 + 256))

69703
69704
69713
69714
69723
69724
69734
________________
69813
69814
69823
69824
69834


Figured out that 31 down must be 213, otherwise 37 across plus 6 down starts 698, which we can't do because the 8's are used up in 24 and 26 across positions 3 and 4.

## 32 and 33 down

In [29]:
print(list(multiples_of(213, 200, 1000)))

[213, 426, 639, 852]


## 9. 15-down

In [31]:
candidates_15 = list(filter_for_distinct(list(multiples_of(11, 100000, 1000000))))

candidates_15 = [
    candidate
    for candidate in candidates_15
    if str(candidate)[0] == "3"
    and str(candidate)[1] == "8"
    and str(candidate)[3] == "1"
    and str(candidate)[5] == "5"
]


print(candidates_15)

[382195, 384175, 387145, 389125]


## 10. 25-across and 42-across

In [32]:
candidates_25 = [577 + 100 * n for n in range(10)]

candidates_25 = [candidate + 10 * n for n in [6, 8] for candidate in candidates_25]

candidates_25 = list(
    filter_for_distinct([candidate for candidate in candidates_25 if candidate > 1100])
)

print(candidates_25)

candidates_42 = [candidate - 576 for candidate in candidates_25]
print(candidates_42)

[1237, 1437, 1537, 1257, 1357, 1457]
[661, 861, 961, 681, 781, 881]


## 11. 22-down

In [45]:
candidates_22 = find_primes(100, 1000)

candidates_22 = [
    candidate
    for candidate in candidates_22
    if str(candidate)[2] == "7" and str(candidate)[1] in "35"
]

print(candidates_22)

[137, 157, 257, 337, 457, 557, 757, 857, 937]


Not a lot of commentary here on my end, but he progress just kept rolling, ended up using this sheet more like a calculator than anything.  